In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
charter = pd.read_excel('charter_vessels_dec.xlsx')

In [3]:
sorrento= pd.read_excel('sorrento_vessels_dec.xlsx')

In [17]:
msc_sm = pd.read_excel('sm_vessels.xlsx')

In [6]:
lima_sm = pd.read_excel(r'D:\Navalt\MSC\2024\Weekly Data\CLIENT\Actual data by client\LimassolExport - 2024 Week 51.xlsx')
lima_sm.shape

(375069, 161)

In [7]:
archived= pd.read_excel("archived_vessels_dec.xlsx")

### Active SM Vessels Checking & Missing Vessels Reporting

# IMO WITH VNAME


In [19]:
# Extract the IMO numbers and vessel names for each dataset
charter_dict = dict(zip(charter['IMO'], charter['Name']))
sorrento_dict = dict(zip(sorrento['IMO'], sorrento['Name']))
sm_dict = dict(zip(msc_sm['IMO'], msc_sm['Name']))
lima_dict = dict(zip(lima_sm['ImoNumber'], lima_sm['vName']))
archived_dict = dict(zip(archived['IMO'], archived['Name']))

# Convert the sets of IMOs to lists for comparison
charter_vessels = set(charter['IMO'])
sorrento_vessels = set(sorrento['IMO'])
sm_vessels = set(msc_sm['IMO'])
lima_vessels = set(lima_sm['ImoNumber'])
archived_vessels = set(archived['IMO'])

# Compare SM vessels with Limassol data
missing_sm_in_lima = [imo for imo in (sm_vessels - lima_vessels)]  # SM vessels missing in Limassol
present_sm_in_lima = [(imo, sm_dict[imo]) for imo in (sm_vessels & lima_vessels)]  # SM vessels present in Limassol

# Check Charter and Sorrento vessels in Limassol data
charter_in_lima = [(imo, charter_dict[imo]) for imo in (charter_vessels & lima_vessels)]  # Charter vessels in Limassol
sorrento_in_lima = [(imo, sorrento_dict[imo]) for imo in (sorrento_vessels & lima_vessels)]  # Sorrento vessels in Limassol

# Check Archived vessels in Limassol data
archived_in_lima = [(imo, archived_dict[imo]) for imo in (archived_vessels & lima_vessels)]  # Archived vessels in Limassol

# Create a DataFrame to store the results
result_df = pd.DataFrame({
    'SM Vessels in Limassol': pd.Series([f"{imo} - {name}" for imo, name in present_sm_in_lima]),
    'SM Vessels Missing in Limassol': pd.Series([f"{imo} - {sm_dict[imo]}" for imo in missing_sm_in_lima if imo in sm_dict]),
    'Charter Vessels in Limassol': pd.Series([f"{imo} - {name}" for imo, name in charter_in_lima]),
    'Sorrento Vessels in Limassol': pd.Series([f"{imo} - {name}" for imo, name in sorrento_in_lima]),
    'Archived Vessels in Limassol': pd.Series([f"{imo} - {name}" for imo, name in archived_in_lima])
})

# Export the results to an Excel file
result_df.to_excel('vessel_comparison_results.xlsx', index=False)

print("Comparison complete. Results saved to 'vessel_comparison_results.xlsx'.")


Comparison complete. Results saved to 'vessel_comparison_results.xlsx'.


In [ ]:
#removal of charter vesssel 

In [ ]:
# IMO numbers to remove
imo_to_remove = [9618305, 9143879, 9256365, 9618317, 9135638, 9618264, 9618290]

lima = lima_sm[~lima_sm['ImoNumber'].isin(imo_to_remove)]

print(lima.shape)


### Missing Dates Reporting

# Month wise missing date

In [ ]:
#client infor sharing

In [ ]:
import pandas as pd

# Assuming 'lima_sm' is your DataFrame with the relevant data
lima_sm = lima  # Replace with your actual DataFrame name
lima_sm_vessels = set(lima_sm['ImoNumber'].unique())  # Define the IMO list

# Determine the latest month in the data
latest_month = pd.to_datetime(lima_sm['DateTimeUtc']).max().strftime('%b-%Y')

# Initialize dictionaries for missing dates tracking
missing_dates_dict = {}
longest_consecutive_missing = {}  # Store the longest consecutive missing dates for each IMO
vessels_with_high_missing = []  # List to store vessels with more than 30 missing dates

# Dictionary to store the vessel with the most missing dates for each month
most_missing_vessel_per_month = {}

# Create a dictionary to store the sum of missing days for each month
monthly_missing_days = {}

# Loop through each IMO in your vessel list
for imo in lima_sm_vessels:
    data = lima_sm[lima_sm['ImoNumber'] == imo]
    
    # Convert 'DateTimeUtc' to datetime and get unique reported dates
    reported_dates = pd.to_datetime(data['DateTimeUtc'], dayfirst=True).dt.date.unique()
    
    if len(reported_dates) > 0:
        start_date = reported_dates.min()
        end_date = reported_dates.max()
        
        # Generate a complete date range
        date_range = pd.date_range(start=start_date, end=end_date).date
        
        # Identify missing dates
        missing = [date for date in date_range if date not in reported_dates]
        
        # Store missing dates in the dictionary, excluding the latest month
        if missing:
            for date in missing:
                month_year = date.strftime('%b-%Y')  # Format as "Jan-2024"
                if month_year == latest_month:
                    continue  # Skip the latest month
                if month_year not in missing_dates_dict:
                    missing_dates_dict[month_year] = {}
                if imo not in missing_dates_dict[month_year]:
                    missing_dates_dict[month_year][imo] = []
                missing_dates_dict[month_year][imo].append(date)
        
        # Calculate the longest sequence of consecutive missing dates
        if missing:
            sorted_missing = sorted(missing)
            max_length = 1
            current_length = 1
            
            for i in range(1, len(sorted_missing)):
                if (sorted_missing[i] - sorted_missing[i - 1]).days == 1:
                    current_length += 1
                    max_length = max(max_length, current_length)
                else:
                    current_length = 1
            
            longest_consecutive_missing[imo] = max_length

        # Identify vessels with more than 30 missing dates
        if len(missing) >= 30:
            vessels_with_high_missing.append((imo, len(missing)))
        
        # Find the vessel with the most missing dates for each month
        for month_year, imos in missing_dates_dict.items():
            for vessel_imo, dates in imos.items():
                if month_year not in most_missing_vessel_per_month:
                    most_missing_vessel_per_month[month_year] = (vessel_imo, len(dates))
                else:
                    if len(dates) > most_missing_vessel_per_month[month_year][1]:
                        most_missing_vessel_per_month[month_year] = (vessel_imo, len(dates))

# Calculate the total missing days per month
for month_year, imos in missing_dates_dict.items():
    total_missing_days = 0  # Initialize a total for this month
    
    # Loop through the vessels in the current month and sum the missing days
    for imo, missing_dates in imos.items():
        total_missing_days += len(missing_dates)  # Add the missing days count for this vessel
    
    # Store the total missing days for this month
    monthly_missing_days[month_year] = total_missing_days

# Find the month with the highest total missing days
month_with_highest_missing = max(monthly_missing_days.items(), key=lambda x: x[1])

# Find the vessel with the highest number of missing dates
imo_with_highest_missing = max(vessels_with_high_missing, key=lambda x: x[1]) if vessels_with_high_missing else None

# Find the vessel with the longest consecutive missing dates
vessel_with_longest_consecutive = max(longest_consecutive_missing.items(), key=lambda x: x[1]) if longest_consecutive_missing else (None, None)

# Output the results
print(f"The month with the highest total missing dates is: {month_with_highest_missing[0]} with {month_with_highest_missing[1]} missing dates.")
print(f"The vessel with IMO {imo_with_highest_missing[0]} has the highest number of missing dates: {imo_with_highest_missing[1]}.")
print(f"The vessel with the longest consecutive missing dates is IMO {vessel_with_longest_consecutive[0]} with {vessel_with_longest_consecutive[1]} consecutive days missing.")

z# Now we need to create two summary tables for the Excel sheet

# Summary 1: Total Missing Days per Month
summary_data_1 = [
    (month, total_missing_days)  # Store month and total missing days
    for month, total_missing_days in monthly_missing_days.items()
]

summary_df_1 = pd.DataFrame(summary_data_1, columns=['Month-Year', 'Total Missing Days'])

summary_df_1['Month-Year'] = pd.to_datetime(summary_df_1['Month-Year'], format='%b-%Y')

summary_df_1 = summary_df_1.sort_values('Month-Year')

summary_df_1['Month-Year'] = summary_df_1['Month-Year'].dt.strftime('%b-%Y')

summary_data_2 = [
    (month, imo, count)  # Format Month-Year, IMO, and Missing Days count
    for month, (imo, count) in most_missing_vessel_per_month.items()
]

# Create a DataFrame for Summary 2
summary_df_2 = pd.DataFrame(summary_data_2, columns=['Month-Year', 'IMO', 'Number of Missing Days'])

# Convert 'Month-Year' column to datetime for sorting
summary_df_2['Month-Year'] = pd.to_datetime(summary_df_2['Month-Year'], format='%b-%Y')

# Sort by 'Month-Year'
summary_df_2 = summary_df_2.sort_values('Month-Year')

summary_df_2['Month-Year'] = summary_df_2['Month-Year'].dt.strftime('%b-%Y')

with pd.ExcelWriter('missing_dates_report.xlsx') as writer:
    summary_df_1.to_excel(writer, sheet_name='Total Missing Days', index=False)
    
    summary_df_2.to_excel(writer, sheet_name='Most Missing IMO', index=False)
    
    # Write the missing dates data for each month to separate sheets
    for month_year, imos in missing_dates_dict.items():
        month_df = pd.DataFrame(
            [(imo, len(dates), ', '.join(map(str, dates))) for imo, dates in imos.items()],
            columns=['IMO', 'Missing Count', 'Missing Dates']
        )
        
        # Write to the corresponding month sheet
        month_df.to_excel(writer, sheet_name=month_year, index=False)

print("Missing dates report created with two summary sheets and separate sheets for each month in 'MMM-YYYY' format.")


# SEQUENCE MISTAKE MONTH WISE

In [ ]:
df=lima.copy()

In [ ]:
a = ['Noon', 'Port Noon', 'EOSP', 'Sailing', 'COSP', 'Berthing', 'Delay', 'COSP Transit']
df = df[df['MessageType'].isin(a)]


In [ ]:
valid_transitions = {
    'Berthing': ['Port Noon', 'Sailing'],
    'Port Noon': ['Port Noon', 'Sailing'],
    'Sailing': ['Delay', 'COSP'],
    'COSP': ['Noon', 'EOSP'],
    'Noon': ['Noon', 'EOSP'],
    'EOSP': ['COSP Transit', "COSP", 'Delay', 'Berthing'],
    'Delay': ['Delay', 'COSP Transit', 'COSP', 'Berthing', 'Noon', 'EOSP'],
    'COSP Transit': ['Noon', 'EOSP']
}

# Function to check for invalid transitions and include the previous row
def check_invalid_transitions(group):
    prev_row = None
    invalid_transitions_data = []

    # Iterate over the DataFrame group to check the order of message types
    for index, row in group.iterrows():
        message_type = row['MessageType']

        # Check if the current message type is valid based on the previous one
        if prev_row is not None:
            prev_message_type = prev_row['MessageType']
            if message_type not in valid_transitions.get(prev_message_type, []):
                # Store both the previous row and the invalid transition row
                invalid_transitions_data.append(prev_row.to_dict())
                invalid_transitions_data.append(row.to_dict())

        # Update the previous row for the next iteration
        prev_row = row

    # If there are invalid transitions, return a DataFrame with the accumulated data
    if invalid_transitions_data:
        invalid_transitions_df = pd.DataFrame(invalid_transitions_data)
        return invalid_transitions_df
    else:
        return None

# Group the DataFrame by IMO number and vessel name
grouped = df.groupby(['ImoNumber', 'vName'])

# List to store all invalid transitions DataFrames
all_invalid_transitions_dfs = []

for (_, _), group in grouped:
    invalid_transitions_df = check_invalid_transitions(group)
    if invalid_transitions_df is not None:
        all_invalid_transitions_dfs.append(invalid_transitions_df)

# Concatenate all DataFrames into one
combined_invalid_transitions_df = pd.concat(all_invalid_transitions_dfs, ignore_index=True)
# Ensure the date column is in datetime format
combined_invalid_transitions_df['Date'] = pd.to_datetime(combined_invalid_transitions_df['DateTimeUtc'])

# Add a 'Month-Year' column for easy grouping by month
combined_invalid_transitions_df['Month-Year'] = combined_invalid_transitions_df['Date'].dt.to_period('M')

# Calculate unique pairs of invalid transitions
# Calculate unique pairs of invalid transitions
if not combined_invalid_transitions_df.empty:
    # Create pairs of invalid transitions by combining the previous and current message types
    combined_invalid_transitions_df['InvalidPair'] = (
        combined_invalid_transitions_df['MessageType'].shift(1) + ' -> ' + combined_invalid_transitions_df['MessageType']
    )
    
    # Filter the rows to include only the second message of each invalid pair
    invalid_pairs = combined_invalid_transitions_df[~combined_invalid_transitions_df['InvalidPair'].isna()].copy()

    # Remove duplicates to ensure we are counting only unique invalid pairs for each IMO
    unique_pairs = invalid_pairs[['ImoNumber', 'InvalidPair']].drop_duplicates()

    # Find the IMO with the most unique invalid transitions (unique invalid pairs)
    overall_max_imo = unique_pairs['ImoNumber'].value_counts().idxmax()
    overall_max_count = unique_pairs['ImoNumber'].value_counts().max()

    # Print the overall result
    print(f"Most Frequent Invalid Transitions: IMO {overall_max_imo} with {overall_max_count} unique invalid transition pairs")
    print(f"Most Frequent Invalid Transitions: IMO {overall_max_imo}")


    # Calculate monthly summary based on unique pairs
    invalid_pairs['Month-Year'] = invalid_pairs['Date'].dt.strftime('%b-%Y')
    
    # Count unique invalid pairs for each IMO per month
    summary = (
        invalid_pairs.groupby(['Month-Year', 'ImoNumber'])['InvalidPair']
        .nunique()  # Count unique invalid pairs for each IMO per month
        .reset_index(name='Max Invalid Sequence Transitions')
    )

    # For each month, find the IMO(s) with the most unique invalid transition pairs
    monthly_summary = summary.groupby('Month-Year').apply(lambda x: x[x['Max Invalid Sequence Transitions'] == x['Max Invalid Sequence Transitions'].max()]).reset_index(drop=True)

    # Convert 'Month-Year' for better readability in Excel
    monthly_summary['Month-Year'] = pd.to_datetime(monthly_summary['Month-Year'], format='%b-%Y')
    monthly_summary = monthly_summary.sort_values(by='Month-Year')
    monthly_summary['Month-Year'] = monthly_summary['Month-Year'].dt.strftime('%b-%Y')

    # Write results to Excel
    with pd.ExcelWriter('Invalid_Transitions_Monthly_Report.xlsx') as writer:
        # Write the summary to the first sheet
        monthly_summary.to_excel(writer, sheet_name='Summary', index=False)

        # Group by 'Month-Year' and save each group as a separate sheet
        for month_year, group in combined_invalid_transitions_df.groupby('Month-Year'):
            group = group.sort_values(by='Date')
            group.to_excel(writer, sheet_name=month_year.strftime('%b-%Y'), index=False)

    print("Summary and detailed invalid transitions written to the Excel file.")
else:
    print("No invalid transitions found.")




### Outlier Reporting

In [ ]:
def find_outliers(data, column, lower_bound, upper_bound):
    outliers = data[(data[column] > upper_bound) | (data[column] < lower_bound)][['ImoNumber', 'vName', 'DateTimeUtc', column]].copy()
    outliers['outlier'] = column
    outliers.rename(columns={column: 'value'}, inplace=True)
    return outliers

# List of columns with their bounds and labels
outlier_conditions = [
    ('CargoMT', 0, 300000),
    ('CargoTEU', 0, 25000),
    ('DraftAft', 0, 30),
    ('DraftFwd', 0, 30),
    ('Speed by gps', 0, 50),
    ('Speed by log', 0, 50),
    ('Power_AE_1', 0, 7000),
    ('Power_AE_2', 0, 7000),
    ('Power_AE_3', 0, 7000),
    ('Power_AE_4', 0, 7000),
    ('Power_AE_5', 0, 7000),
    ('Power_ME', 0, 100000),
    ('RPM', 0, 750),
    ('Miles by gps', 0, 1000),
    ('Slip', -100, 100),
    ('Sea state', 0, 10),
    ('Hours_AE_1', 0, 26),
    ('Hours_AE_2', 0, 26),
    ('Hours_AE_3', 0, 26),
    ('Hours_AE_4', 0, 26),
    ('Hours_AE_5', 0, 26),
    ('Hours_ME', 0, 26),
    ('Hours_SG', 0, 26),
    ('Hours_TG', 0, 26)
]

outliers_list = [find_outliers(lima, col, lb, ub) for col, lb, ub in outlier_conditions]

all_outliers = pd.concat(outliers_list, ignore_index=True)

outliers = all_outliers[['ImoNumber', 'vName', 'DateTimeUtc', 'outlier', 'value']]


outliers

In [ ]:
# Count the occurrences of each outlier type and get the most frequent one
outlier_counts = outliers['outlier'].value_counts()

# Find the outlier with the maximum frequency
most_frequent_outlier = outlier_counts.idxmax()
most_frequent_outlier_count = outlier_counts.max()

# Print the summary
print("Summary of Outliers Detected Across Vessel Operations:")

print(f"1. A total of {len(outliers)} outlier instances were identified across all vessels.")

print(f"2. The most frequent outlier was '{most_frequent_outlier}' with {most_frequent_outlier_count} instances.")


In [ ]:
# List of parameters to check for the highest value
parameters = [
    'CargoMT', 'CargoTEU', 'DraftAft', 'DraftFwd', 'Speed by gps', 'Speed by log',
    'Power_AE_1', 'Power_AE_2', 'Power_AE_3', 'Power_AE_4', 'Power_AE_5', 'Power_ME',
    'RPM', 'Miles by gps', 'Slip', 'Sea state', 'Hours_AE_1', 'Hours_AE_2', 'Hours_AE_3',
    'Hours_AE_4', 'Hours_AE_5', 'Hours_ME', 'Hours_SG', 'Hours_TG'
]

# Find the highest value for each parameter
highest_values = {}
for param in parameters:
    if param in lima.columns:
        highest_value_row = lima.loc[lima[param] == lima[param].max()]
        highest_values[param] = {
            'Value': highest_value_row[param].values[0],
            'IMO Number': highest_value_row['ImoNumber'].values[0],
            'Vessel Name': highest_value_row['vName'].values[0],
            'DateTime': highest_value_row['DateTimeUtc'].values[0]
        }

# Print the highest values for each parameter
print("Summary of Highest Values Across Parameters:")

for param, details in highest_values.items():
    print(f"1. Highest value for '{param}':")
    print(f"    - Value: {details['Value']}")
    print(f"    - IMO Number: {details['IMO Number']}")
    print(f"    - Vessel Name: {details['Vessel Name']}")
    print()


In [ ]:
outliers.to_csv('outliers.csv', index=False)

# Duplication  

In [ ]:
df=lima

In [ ]:
df.columns.to_list()

In [ ]:
import pandas as pd


def check_duplicates(group):
    duplicate_utc = group[group.duplicated(subset='DateTimeUtc', keep=False)]
    
    duplicate_utc_same_message = duplicate_utc[duplicate_utc.duplicated(subset=['DateTimeUtc', 'MessageType'], keep=False)]
    
    return duplicate_utc_same_message

grouped_duplicates = df.groupby('ImoNumber').apply(check_duplicates)

new_df = grouped_duplicates.reset_index(drop=True)


print(new_df)  


In [ ]:
import pandas as pd

def check_duplicates(group):
    # Identify rows with duplicate DateTimeUtc values
    duplicate_utc = group[group.duplicated(subset='DateTimeUtc', keep=False)]
    
    duplicate_utc_same_message = duplicate_utc[duplicate_utc.duplicated(subset=['DateTimeUtc', 'MessageType'], keep=False)]
    
    if not duplicate_utc_same_message.empty:
        refueling_rows = duplicate_utc_same_message[duplicate_utc_same_message['MessageType'] == 'Refueling']
        if not refueling_rows.empty:
            # Remove rows where Volume or Quantity are not identical for the same DateTimeUtc
            valid_refueling_rows = refueling_rows[refueling_rows.duplicated(subset=['DateTimeUtc', 'Volume', 'Quantity', 'Bdn number', 'Product Name', 'Physical supplier name','Tanktainer number'], keep=False)]
            # Combine non-refueling and valid refueling rows
            non_refueling_rows = duplicate_utc_same_message[duplicate_utc_same_message['MessageType'] != 'Refueling']
            duplicate_utc_same_message = pd.concat([non_refueling_rows, valid_refueling_rows], ignore_index=True)

    return duplicate_utc_same_message

grouped_duplicates = df.groupby('ImoNumber').apply(check_duplicates)

# Reset the index for the resulting DataFrame
new_df = grouped_duplicates.reset_index(drop=True)

print(new_df)


In [ ]:
new_df.to_excel('Duplicates-1.xlsx')

# Vessel Name checking

In [ ]:
lima_sm

In [ ]:
msc_sm

In [ ]:
lima_sm.rename(columns={'ImoNumber': 'IMO'}, inplace=True)
lima_sm


In [ ]:
lima_sm['IMO'] = lima_sm['IMO'].astype(str)
msc_sm['IMO'] = msc_sm['IMO'].astype(str)


In [ ]:
# Normalize and check for hidden discrepancies between VNAME and NAME
mismatched_vessels['VNAME_normalized'] = mismatched_vessels['VNAME'].str.strip().str.lower()
mismatched_vessels['NAME_normalized'] = mismatched_vessels['NAME'].str.strip().str.lower()


In [ ]:
# Merge the two dataframes on 'IMO'
merged_df = pd.merge(lima_sm, msc_sm, on='IMO', how='inner')


In [ ]:
unique_mismatched_vessels = final_mismatched_vessels[['IMO', 'vName', 'Name']].drop_duplicates()

print(unique_mismatched_vessels)

In [ ]:
unique_mismatched_vessels.to_csv(r'C:\Users\akshaya\Desktop\mismatched_vessels.csv', index=False)

In [ ]:
#take over

In [29]:
takeover_dates=pd.read_excel('Takeoverdates.xlsx')
takeover_dates

,IMO Number,Fleet,Ship Name,dates
0,9116589,4,MSC UNITE VI,2024-11-18
1,9289556,5,MSC MANHATTAN V,2024-11-13
2,9445899,3,MSC CALIDRIS III,2024-11-11
3,9221815,12,MSC KERRY VII,2024-10-24
4,9409041,6,MSC WEST V,2024-10-23
...,...,...,...,...
159,9223904,12,MSC SUN F,2023-01-18
160,9932892,16,MSC MARA,2023-01-11
161,9244946,19,MSC KALAMATA VII,2023-01-09
162,9344708,21,MSC GENERAL IV,2023-01-05


In [31]:
lima_imos = lima_sm['ImoNumber'].dropna().unique()  
takeover_imos = takeover_dates['IMO Number'].dropna().unique() 


In [33]:

# Find vessels in takeover_dates but not in lima
missing_imos = set(takeover_imos) - set(lima_imos)

# Filter takeover_dates to get the missing vessels
missing_vessels = takeover_dates[takeover_dates['IMO Number'].isin(missing_imos)]

# Print the result
print(f"Number of missing vessels: {len(missing_imos)}")
missing_vessels

Number of missing vessels: 73


,IMO Number,Fleet,Ship Name,dates
0,9116589,4,MSC UNITE VI,2024-11-18
1,9289556,5,MSC MANHATTAN V,2024-11-13
2,9445899,3,MSC CALIDRIS III,2024-11-11
3,9221815,12,MSC KERRY VII,2024-10-24
4,9409041,6,MSC WEST V,2024-10-23
...,...,...,...,...
72,9932921,22,MSC ADYA,2024-02-01
74,9957359,22,MSC ROSE,2024-01-20
75,9962562,8,MSC IDANIA,2024-01-18
80,9360764,21,MSC BASEL V,2023-12-21


In [37]:
takeover_dates['dates'] = pd.to_datetime(takeover_dates['dates'], errors='coerce')

# Filter rows where the year is 2024
takeover_2024 = takeover_dates[takeover_dates['dates'].dt.year == 2024]

lima_imos = lima_sm['ImoNumber'].dropna().unique()
takeover_2024_imos = takeover_2024['IMO Number'].dropna().unique()

missing_imos_2024 = set(takeover_2024_imos) - set(lima_imos)
missing_vessels_2024 = takeover_2024[takeover_2024['IMO Number'].isin(missing_imos_2024)]


# Print the result
print(f"Number of missing vessels in 2024: {len(missing_imos_2024)}")
missing_vessels_2024

Number of missing vessels in 2024: 71


,IMO Number,Fleet,Ship Name,dates
0,9116589,4,MSC UNITE VI,2024-11-18
1,9289556,5,MSC MANHATTAN V,2024-11-13
2,9445899,3,MSC CALIDRIS III,2024-11-11
3,9221815,12,MSC KERRY VII,2024-10-24
4,9409041,6,MSC WEST V,2024-10-23
...,...,...,...,...
69,9173135,7,MSC WIND II,2024-02-21
71,9946879,24,MSC CARMELITA,2024-02-08
72,9932921,22,MSC ADYA,2024-02-01
74,9957359,22,MSC ROSE,2024-01-20
